In [214]:
# Importing required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random
warnings.filterwarnings("ignore")

In [215]:
# Train test split
def train_test_split(data, test_size):
    
    if isinstance(test_size, float):
        test_size = round(test_size*len(data))
        
    indices = data.index.tolist()  # random.sample take list, set, dictionary
    test_indices =random.sample(population = indices, k=test_size)
    
    test = data.loc[test_indices]
    train = data.drop(test_indices)
    
    return train, test

In [216]:
# Calculate Prior Probabilities
def calculate_prior_probabilities(data):
    
    prior_probabilities = data.groupby(by = 'target').apply(lambda x: len(x)/len(data))
    
    return np.log(prior_probabilities).values

In [217]:
# Find mean, varience
def return_statistics(data):
    
    mean = data.groupby(by='target').apply(lambda x: x.mean(axis=0))
    variance = data.groupby(by='target').apply(lambda x: x.var(axis=0))
    
    return (mean.values, variance.values)

In [218]:
# Find Gaussian Probability density

# P(x=12 | 'setosa')

def calculate_probability_density(mean, variance, x):
    
    probability_density = (1 / np.sqrt(2*np.pi*variance)) * np.exp( (-(x - mean) ** 2) / (2 * variance))
    
    return probability_density

In [219]:
# Posterior Probabilities
def calculate_posterior_probabilities(data_row, mean, variance, n_unique_labels, n_cols):
    
    posterior_probabilities = []
    
    # calculate probabilities wrt each label to find max
    for i in range(n_unique_labels):
        posterior = 0
        
        #for each feature
        for j in range(n_cols):
            posterior += np.log(calculate_probability_density(mean[i][j], variance[i][j], data_row[j]))
        posterior_probabilities.append(posterior)
        
    return posterior_probabilities

In [220]:
# Fit model on training dataset

def NBA_fit(data):
    
    n_cols = len(data.columns)-1
    unique_labels = data['target'].unique()
    n_unique_labels = len(unique_labels)
    
    mean, variance = return_statistics(data)
    prior_probabilities = calculate_prior_probabilities(data)  # returns log
    
    return {
        'n_cols': n_cols,
        'unique_labels': unique_labels,
        'n_unique_labels': n_unique_labels,
        'mean' : mean,
        'variance' : variance,
        'prior_probabilities' : prior_probabilities
    }

In [221]:
# Predict
def predict(test_data, nba):
    
    predictions = []
    for i in range(len(test_data)):
        
        prior = nba['prior_probabilities']
        posterior = calculate_posterior_probabilities(test_data.iloc[i, :-1], nba['mean'], nba['variance'], nba['n_unique_labels'], nba['n_cols'])
        
        probabilities = prior + posterior
        # one with max prob will be the output
        mx_idx = np.argmax(probabilities)
        
        predictions.append(nba['unique_labels'][mx_idx])  # add log values
        
    return predictions

In [222]:
data = sns.load_dataset('iris')
data_copy = data.copy()
data.rename(columns = {'species' : 'target'}, inplace = True)

data.head()

,sepal_length,sepal_width,petal_length,petal_width,target
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [223]:
# train test split
train_data, test_data = train_test_split(data, 0.2)

# fit model 
nba = NBA_fit(train_data)

# make predictions
predictions = predict(test_data, nba)

# accuracy
accuracy = len(test_data.loc[predictions == test_data['target']])/len(test_data) * 100

accuracy

93.33333333333333